# Inner speech: reconstruction of the paper's methods

**Status: runnable methods reconstruction, NOT a verified reproduction of the reported accuracy.**

Reference: Tasin et al., *Ensemble Machine Learning Model for Inner Speech Recognition:
A Subject-Specific Investigation*, IEEE Access 14, 10811–10827 (2026),
[DOI 10.1109/ACCESS.2025.3644494](https://doi.org/10.1109/ACCESS.2025.3644494).
This is a new, self-contained notebook. The previous EEGNet/LaBraM notebook is separate.

The attached journal PDF references Supplementary Tables T1–T3 for the exact 191 feature
definitions, but those tables were not available in the supplied material and could not be
retrieved from the publisher. The public [preprint](https://arxiv.org/abs/2412.17824) also
references a separate supplement. It would be incorrect to invent 191 features and call
them the paper's feature bank.

This notebook therefore provides:

- Processed-dataset loading with event alignment, units, labels and timing checks.
- Mean/std artifact inspection, six-mode VMD and removal of the lowest-frequency mode.
- An explicitly defined **152-feature reconstruction** of the described families
  (50 temporal/envelope + 24 spectral + 78 wavelet). These definitions are assumptions.
- A validated import route for the authors' exact **191-feature tensor**, when obtained.
- MRMR with the paper's relevance/redundancy **quotient** form, selecting 590 features.
- Six classical classifiers, three ensemble variants, a two-layer 1D CNN,
  and feature-image ResNet18/VGG16.
- Fresh models for every fold, exact out-of-fold subject metrics, saved model bundles,
  data/configuration hashes and resume support.

**No result from `documented_152` should be described as reproducing the 191-feature paper.**
More features, VMD, or matching a reported number do not guarantee higher accuracy.
The previous performance gap has not yet been causally attributed to these differences.

## What matches, and what remains an assumption

| Stage | Implemented | Status |
|---|---|---|
| Dataset | Inner speech, 4 labels, 128 BioSemi electrodes, processed 256 Hz FIF | Same dataset family; content hashes identify the actual mirror |
| Provider preprocessing | Use existing rereferenced, 0.5–100 Hz/50 Hz notch/ICA derivatives | Not reapplied; a FIF header alone cannot certify historical ICA |
| Action interval | `[1.0,3.5)` seconds using `epochs.times`; 640 samples at 256 Hz | Matches the dataset publication's trial workflow; inspect timing for your derivative version |
| Artifact detection | Per-trial robust mean/std rule or manually supplied electrode names | Authors' inspected channels and thresholds unavailable |
| VMD | Six modes; discard mode with smallest final center frequency | Alpha=2000, tau=0, DC=0, init=1, tol=1e-7 and full-epoch application are assumptions |
| Features | 152 defined features; alternative import of verified 191-feature arrays | Exact T1–T3 extraction is still unresolved |
| MRMR | Full candidate pool, MI relevance / mean absolute Pearson correlation | Ratio matches Eq.1; library/estimators and correlation floor unspecified |
| Feature count | K=590 fixed from the publication | Do not retune K on held-out subject results |
| SVM | CPU scikit-learn SVC, default RBF C=1 gamma=scale; optional nested search | Paper's exact hyperparameters were not disclosed |
| Other ML | LDA, Extra Trees, Random Forest, LR, LightGBM | Model families match; settings are recorded assumptions |
| Ensembles | 3 LR, 5 LR, and SVM+ETC+LR probability voting | Five-LR bootstraps and soft voting follow Methods; specific settings assumed |
| CNN | 1D: kernel3, pool4, dropout0.1 and MLP dropout0.5 | Channel widths/MLP size/training settings assumed; S2 unavailable |
| Image CNNs | ResNet18/VGG16 on feature×channel images, specified rotations/translations | Scaling/rendering and initialization assumed; all augmented versions stay in train |
| Evaluation | Subject-specific 10 folds for ML/1D CNN, 5 for image CNNs | Follows the explicit Evaluation Criteria section; another paragraph is less precise |
| Scoring | Concatenate OOF trials per subject, then average subjects | Matches the stated aggregation; macro and weighted F1 both saved since averaging is unspecified |
| Channel reduction | Optional nested 64/32/16 subsets | Extension; begin with the paper's 128-channel baseline |

The preliminary eight-selector comparison, exploratory K=20–610 sweep, and ERP figures
are not reproduced here. This notebook implements the selected MRMR workflow and model
evaluation, not every exploratory analysis in the article.

## 1. Dependencies

Kaggle already supplies NumPy, SciPy, pandas, scikit-learn and PyTorch. Uncomment the first
line only if the additional packages are missing. Keep Kaggle's matched torch/torchvision
installation. Restart the kernel after installing packages if an import reports an ABI issue.

VMD, feature engineering, MRMR and all classical models run on CPU. A T4 accelerates the
CNNs when `torch.cuda.is_available()` is true. This notebook intentionally does not require
RAPIDS/cuML; it uses the same SVC implementation for tuning, final fitting and checkpoints.

In [ ]:
# %pip install "mne>=1.6,<2" "vmdpy==0.2" "PyWavelets>=1.4,<2" "lightgbm>=4,<5" cloudpickle tqdm
# Tested core environment: numpy 1.26.4, scipy 1.11.4, sklearn 1.4.2, mne 1.6.1.
# A compatible recent Kaggle environment is also intended; package versions are recorded.

## Imports and configuration

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field, asdict
from pathlib import Path
import hashlib
import importlib.metadata
import inspect
import json
import os
import random
import re
import warnings
import time
import numpy as np
import pandas as pd
import scipy
from scipy import signal, stats
from scipy.integrate import trapezoid
import pywt
import joblib
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, VotingClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, ParameterGrid
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from tqdm.auto import tqdm

METHOD_VERSION = "paper-methods-1.0"
CLASS_NAMES = ("Up", "Down", "Right", "Left")
MISSING_AUTHOR_DETAILS = [
    "Supplementary T1-T3: exact definitions of all 191 features, wavelets and levels",
    "Artifact channel identities/detection thresholds and VMD alpha/tau/DC/init/tolerance",
    "MRMR library, relevance estimator and placement relative to cross-validation",
    "Classifier hyperparameters, original folds/seeds and multiclass F1 averaging",
    "Supplementary S2: CNN widths; optimizer, epochs and image-rendering settings",
    "Ensemble description calls it stacking but specifies soft probability voting",
]


@dataclass
class PaperConfig:
    data_root: str = "/kaggle/input/datasets/truthisneverlinear/inner-speech-recognition/inner-speech-recognition/derivatives"
    output_root: str = "/kaggle/working/paper_methods"
    subject_ids: tuple | None = (1,)  # None = all available subjects
    channel_counts: tuple = (128,)
    seeds: tuple = (42,)
    ml_models: tuple = ("svm", "lr", "lr5")
    deep_models: tuple = ()  # cnn1d, resnet18, vgg16
    outer_folds: int = 10
    image_outer_folds: int = 5
    inner_folds: int = 3
    tuning: str = "fixed"  # fixed assumptions, or nested hyperparameter search
    selection_scope: str = "fold"  # or diagnostic_pre_cv (optimistic, NOT valid test estimate)
    k_features: int = 590
    mrmr_relevance: str = "mi"  # estimator unspecified in paper; optional f_stat
    mrmr_corr_floor: float = 1e-6
    feature_backend: str = "documented_152"  # or author_191_npz
    author_feature_dir: str | None = None
    action_start: float = 1.0
    action_stop: float = 3.5
    expected_sfreq: float = 256.0
    # Derivatives already include rereferencing, 0.5-100 Hz, 50 Hz notch and ICA.
    # Do NOT repeat those steps on the prepared FIF files.
    artifact_policy: str = "robust_trial"  # manual, all, none also supported
    manual_artifact_channels: dict = field(default_factory=dict)
    artifact_mad_z: float = 4.0  # implementation assumption, not a published threshold
    vmd_scope: str = "full_epoch"  # action also supported; paper is not explicit
    vmd_alpha: float = 2000.0
    vmd_tau: float = 0.0
    vmd_k: int = 6  # specified by paper
    vmd_dc: int = 0
    vmd_init: int = 1  # deterministic uniformly spaced mode initialization
    vmd_tol: float = 1e-7
    wavelet: str = "db4"  # assumption pending T3
    wavelet_level: int = 5  # six coefficient arrays -> 78 documented TFD features
    spectral_high: float = 100.0
    n_jobs: int = 2
    resume: bool = True
    deep_epochs: int = 60
    deep_patience: int = 12
    deep_validation_fraction: float = 0.15
    deep_batch_size: int = 16
    deep_lr: float = 1e-3
    cnn_widths: tuple = (32, 64)
    cnn_hidden: int = 128
    image_size: int = 224
    image_pretrained: bool = False  # pretrained status not specified by paper

    def validate(self):
        if self.selection_scope not in {"fold", "diagnostic_pre_cv"}:
            raise ValueError("Unknown feature-selection scope")
        if self.tuning not in {"fixed", "nested"}:
            raise ValueError("tuning must be fixed or nested")
        if self.feature_backend not in {"documented_152", "author_191_npz"}:
            raise ValueError("Unknown feature backend")
        if self.feature_backend == "author_191_npz" and not self.author_feature_dir:
            raise ValueError("Set author_feature_dir to verified per-subject feature NPZ files")
        if self.artifact_policy not in {"robust_trial", "manual", "all", "none"}:
            raise ValueError("Unknown artifact policy")
        if self.vmd_scope not in {"full_epoch", "action"}:
            raise ValueError("Unknown VMD scope")
        if self.wavelet_level != 5:
            raise ValueError("documented_152 currently specifies level 5; change its manifest too")
        if not (isinstance(self.channel_counts, (tuple, list)) and self.channel_counts):
            raise ValueError("Use channel_counts=(128,), including the comma")
        if not set(self.channel_counts).issubset({128, 64, 32, 16}):
            raise ValueError("Channel counts must be 128, 64, 32 or 16")
        if not self.seeds or len(set(self.seeds)) != len(self.seeds):
            raise ValueError("Supply one or more unique seeds as a tuple")
        if self.subject_ids is not None and not self.subject_ids:
            raise ValueError("subject_ids must be None or a nonempty tuple")
        if len(set(self.channel_counts)) != len(self.channel_counts):
            raise ValueError("Duplicate channel counts")
        if not self.ml_models and not self.deep_models:
            raise ValueError("Enable at least one model")
        if self.k_features < 1:
            raise ValueError("k_features must be positive")
        if self.action_stop <= self.action_start or self.spectral_high > self.expected_sfreq/2:
            raise ValueError("Invalid action interval or spectral range")
        if self.outer_folds < 2 or self.image_outer_folds < 2:
            raise ValueError("Cross-validation requires at least two folds")
        if not 0 < self.deep_validation_fraction < 0.5:
            raise ValueError("Invalid validation fraction")
        if self.selection_scope == "diagnostic_pre_cv" and self.deep_models:
            raise ValueError("Pre-CV selection diagnostic applies to ML only")
        return self


def jsonable(x):
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, np.generic):
        return x.item()
    raise TypeError(type(x).__name__)


def signature(x):
    return hashlib.sha256(json.dumps(x, sort_keys=True, default=jsonable).encode()).hexdigest()[:20]


def atomic_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(obj, indent=2, sort_keys=True, default=jsonable) + "\n")
    temp.replace(path)


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temp, index=False)
    temp.replace(path)


def atomic_npz(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(".tmp")
    with temp.open("wb") as handle:
        np.savez_compressed(handle, **arrays)
    temp.replace(path)


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    except ImportError:
        pass


def versions():
    names = ("numpy", "scipy", "pandas", "scikit-learn", "PyWavelets", "vmdpy",
             "mne", "torch", "torchvision", "lightgbm", "joblib", "cloudpickle")
    out = {}
    for name in names:
        try:
            out[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            out[name] = "not installed"
    return out


def implementation_hash():
    # Notebook builder injects the exact embedded source hash. For the module,
    # hash the current file, including changes that forgot a version bump.
    if "NOTEBOOK_SOURCE_SHA256" in globals():
        return NOTEBOOK_SOURCE_SHA256
    return file_hash(__file__)


def metrics(y, pred):
    return {"accuracy": accuracy_score(y, pred),
            "balanced_accuracy": balanced_accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, labels=range(4), average="macro", zero_division=0),
            "weighted_f1": f1_score(y, pred, labels=range(4), average="weighted", zero_division=0)}

## Dataset loading, alignment and channel geometry

In [ ]:
def discover_recordings(root):
    paths = sorted(Path(root).rglob("*_eeg-epo.fif"))
    if not paths:
        raise FileNotFoundError(f"No *_eeg-epo.fif under {root}. Set cfg.data_root first.")
    recordings = []
    seen = set()
    for path in paths:
        match = re.search(r"sub-(\d+)_ses-(\d+)_eeg-epo\.fif$", path.name)
        if not match:
            continue
        subject, session = map(int, match.groups())
        if (subject, session) in seen:
            raise ValueError(f"Duplicate recording for subject/session {(subject, session)}")
        events_path = path.with_name(path.name.replace("_eeg-epo.fif", "_events.dat"))
        if not events_path.exists():
            raise FileNotFoundError(events_path)
        recordings.append((subject, session, path, events_path))
        seen.add((subject, session))
    if not recordings:
        raise ValueError("Files exist but their names do not match sub-XX_ses-XX_eeg-epo.fif")
    return recordings


def load_subject(cfg, subject):
    import mne
    records = [r for r in discover_recordings(cfg.data_root) if r[0] == subject]
    if not records:
        raise ValueError(f"Subject {subject} is missing")
    canonical = mne.channels.make_standard_montage("biosemi128").ch_names
    blocks, metadata, provenance = [], [], []
    reference_times = None
    for sid, session, path, event_path in records:
        epochs = mne.read_epochs(path, preload=True, verbose=False)
        event_obj = pd.read_pickle(event_path)  # only load your trusted dataset files
        if isinstance(event_obj, pd.DataFrame):
            # Current author repository uses named DataFrame columns.
            colmap = {str(c).lower(): c for c in event_obj.columns}
            label_col = colmap.get("code", colmap.get("label"))
            condition_col = colmap.get("condition")
            if label_col is None or condition_col is None:
                raise ValueError(f"Unrecognized event columns: {list(event_obj.columns)}")
            event_labels = event_obj[label_col].to_numpy()
            conditions = event_obj[condition_col].to_numpy()
        else:
            events = np.asarray(event_obj)
            if events.ndim != 2 or events.shape[1] < 4:
                raise ValueError(f"Unexpected events shape: {events.shape}")
            event_labels, conditions = events[:, 1], events[:, 2]
        if len(event_labels) != len(epochs):
            if len(event_labels) == len(epochs.drop_log):
                event_labels = event_labels[epochs.selection]
                conditions = conditions[epochs.selection]
            else:
                raise ValueError(f"Epoch/event misalignment in {path.name}")
        if not np.isin(event_labels, [0, 1, 2, 3]).all():
            raise ValueError("Expected standardized class labels 0=Up,1=Down,2=Right,3=Left")
        if not np.isin(conditions, [0, 1, 2]).all():
            raise ValueError("Expected standardized conditions 0,1,2")
        sfreq = float(epochs.info["sfreq"])
        if not np.isclose(sfreq, cfg.expected_sfreq):
            raise ValueError(f"Expected processed {cfg.expected_sfreq} Hz derivatives, got {sfreq}")
        missing = set(canonical) - set(epochs.ch_names)
        if missing:
            raise ValueError(f"Missing BioSemi EEG channels: {sorted(missing)}")
        epochs.pick(canonical).reorder_channels(canonical)
        times = epochs.times.copy()
        if reference_times is not None and not np.array_equal(times, reference_times):
            raise ValueError("Different epoch timing across sessions")
        reference_times = times
        action = (times >= cfg.action_start - 1e-9) & (times < cfg.action_stop - 1e-9)
        expected_samples = int(round((cfg.action_stop - cfg.action_start) * sfreq))
        if action.sum() != expected_samples:
            raise ValueError(f"Action window has {action.sum()} samples, expected {expected_samples}")
        chosen = np.flatnonzero(conditions == 1)
        data = epochs.get_data(copy=False)[chosen].astype(np.float64) * 1e6  # volts -> microvolts
        if not np.isfinite(data).all():
            raise ValueError("Non-finite EEG samples; inspect source data")
        blocks.append(data)
        for i in chosen:
            metadata.append({"subject": sid, "session": session,
                             "trial_id": f"sub-{sid:02d}_ses-{session:02d}_row-{int(epochs.selection[i]):04d}",
                             "y_true": int(event_labels[i]), "epoch_row": int(i)})
        provenance.append({"eeg_file": str(path), "eeg_sha256": file_hash(path),
                           "event_file": str(event_path), "event_sha256": file_hash(event_path),
                           "sfreq": sfreq, "highpass": float(epochs.info["highpass"]),
                           "lowpass": float(epochs.info["lowpass"]), "n_inner_trials": len(chosen),
                           "epoch_tmin": float(times[0]), "epoch_tmax": float(times[-1])})
    meta = pd.DataFrame(metadata)
    if meta.trial_id.duplicated().any() or len(meta) == 0:
        raise ValueError("Empty or duplicate trial IDs")
    counts = meta.y_true.value_counts().reindex(range(4), fill_value=0)
    if (counts == 0).any():
        raise ValueError(f"Missing class for subject {subject}: {counts.to_dict()}")
    print(f"Subject {subject:02d}: {len(meta)} inner trials, class counts {counts.to_dict()}")
    return np.concatenate(blocks), reference_times, canonical, meta, provenance


def channel_subsets(names):
    import mne
    pos = mne.channels.make_standard_montage("biosemi128").get_positions()["ch_pos"]
    points = np.asarray([pos[n] for n in names])
    start = np.linalg.norm(points - [0, 0, points[:, 2].max()], axis=1).argmin()
    chosen, remaining = [int(start)], set(range(len(names))) - {int(start)}
    while remaining:
        candidates = np.asarray(sorted(remaining))
        distances = np.linalg.norm(points[candidates, None] - points[chosen][None], axis=2).min(axis=1)
        nxt = int(candidates[np.argmax(distances)])
        chosen.append(nxt)
        remaining.remove(nxt)
    # Full-channel paper baseline retains original BioSemi order for feature images.
    return {128: np.arange(128), **{n: np.array(chosen[:n]) for n in (64, 32, 16)}}

## Motion artifacts: label-blind inspection and VMD

In [ ]:
def robust_z(x):
    median = np.median(x)
    mad = np.median(np.abs(x - median))
    return np.abs(x - median) / max(1.4826 * mad, 1e-12)


def artifact_mask(trial, names, subject, cfg):
    if cfg.artifact_policy == "all":
        return np.ones(len(names), bool)
    if cfg.artifact_policy == "none":
        return np.zeros(len(names), bool)
    if cfg.artifact_policy == "manual":
        selected = cfg.manual_artifact_channels.get(str(subject), cfg.manual_artifact_channels.get(subject))
        if selected is None:
            raise ValueError(f"Supply manual_artifact_channels[{subject}] (empty list if none)")
        unknown = set(selected) - set(names)
        if unknown:
            raise ValueError(f"Unknown artifact channels: {unknown}")
        return np.isin(names, selected)
    # Per-trial across-channel rule uses no label or statistics from other trials.
    # It approximates the paper's visual mean/std inspection, not its exact decisions.
    means = trial.mean(axis=-1)
    log_std = np.log(np.maximum(trial.std(axis=-1), 1e-12))
    return (robust_z(means) > cfg.artifact_mad_z) | (robust_z(log_std) > cfg.artifact_mad_z)


def vmd_remove_lowest(x, cfg):
    from vmdpy import VMD
    x = np.asarray(x, dtype=np.float64)
    if np.std(x) < 1e-12:
        return x.copy(), {"removed_hz": 0.0, "iterations": 0, "constant": True}
    # VMD operates on an even length. Scale for numerical stability and restore units.
    scale = max(np.sqrt(np.mean(x * x)), 1e-12)
    padded = np.pad(x / scale, (0, len(x) % 2), mode="edge")
    modes, _, omega = VMD(padded, cfg.vmd_alpha, cfg.vmd_tau, cfg.vmd_k,
                          cfg.vmd_dc, cfg.vmd_init, cfg.vmd_tol)
    removed = int(np.argmin(omega[-1]))
    reconstructed = modes[np.arange(cfg.vmd_k) != removed].sum(axis=0)[:len(x)] * scale
    if not np.isfinite(reconstructed).all():
        raise ValueError("VMD produced non-finite values")
    return reconstructed, {"removed_hz": float(omega[-1, removed] * cfg.expected_sfreq),
                            "iterations": len(omega), "constant": False}


def prepare_clean_subject(raw, times, names, meta, provenance, cfg):
    selected_keys = ["artifact_policy", "manual_artifact_channels", "artifact_mad_z", "vmd_scope",
                     "vmd_alpha", "vmd_tau", "vmd_k", "vmd_dc", "vmd_init", "vmd_tol",
                     "action_start", "action_stop", "expected_sfreq"]
    cfg_dict = asdict(cfg)
    identity = {"version": METHOD_VERSION, "source_code": implementation_hash(),
                "sources": provenance, "channels": names,
                "preprocess": {k: cfg_dict[k] for k in selected_keys}, "libraries": versions()}
    digest = signature(identity)
    directory = Path(cfg.output_root) / "cache" / f"sub-{int(meta.subject.iloc[0]):02d}_{digest}"
    directory.mkdir(parents=True, exist_ok=True)
    atomic_json(identity, directory / "provenance.json")
    action = (times >= cfg.action_start - 1e-9) & (times < cfg.action_stop - 1e-9)
    source = raw if cfg.vmd_scope == "full_epoch" else raw[:, :, action]
    clean = np.empty((len(raw), len(names), int(action.sum())), dtype=np.float32)
    rows = []
    for index in tqdm(range(len(raw)), desc="VMD + artifact audit"):
        cache = directory / f"trial-{index:04d}.npz"
        if cache.exists() and cfg.resume:
            with np.load(cache, allow_pickle=False) as z:
                clean[index] = z["clean"]
                audit = json.loads(str(z["audit_json"].item()))
            rows.extend(audit)
            continue
        trial = source[index]
        flag = artifact_mask(trial, names, int(meta.subject.iloc[0]), cfg)
        processed = trial.copy()
        audit = []
        for ch, name in enumerate(names):
            diagnostics = {"removed_hz": None, "iterations": 0, "constant": False}
            if flag[ch]:
                processed[ch], diagnostics = vmd_remove_lowest(trial[ch], cfg)
            audit.append({"trial_id": meta.trial_id.iloc[index], "channel": name,
                          "flagged": bool(flag[ch]), "mean_uv": float(trial[ch].mean()),
                          "std_uv": float(trial[ch].std()),
                          "variance_before": float(trial[ch].var()),
                          "variance_after": float(processed[ch].var()), **diagnostics})
        clean[index] = processed[:, action] if cfg.vmd_scope == "full_epoch" else processed
        atomic_npz(cache, clean=clean[index], audit_json=np.array(json.dumps(audit)))
        rows.extend(audit)
    audit_frame = pd.DataFrame(rows)
    atomic_csv(audit_frame, directory / "artifact_audit.csv")
    atomic_csv(meta, directory / "trials.csv")
    return clean, directory, audit_frame


def plot_artifact_audit(audit):
    import matplotlib.pyplot as plt
    by_channel = audit.groupby("channel", sort=False).agg(
        mean_uv=("mean_uv", "mean"), std_uv=("std_uv", "mean"), flagged=("flagged", "mean"))
    fig, axes = plt.subplots(3, 1, figsize=(16, 7), sharex=True)
    for ax, column in zip(axes, by_channel.columns):
        ax.plot(by_channel.index, by_channel[column])
        ax.set_ylabel(column)
    axes[-1].tick_params(axis="x", rotation=90, labelsize=6)
    fig.tight_layout()
    return fig

## Feature engineering: explicit definitions, no claim of exact T1-T3

The 25 temporal statistics are computed on both the signal and its Hilbert envelope.
The spectral bank uses Welch PSD over 0.5–100 Hz, shape descriptors and five band powers.
The wavelet bank uses `db4`, five decomposition levels, and 13 descriptors on each of
`A5,D5,D4,D3,D2,D1`: **78 wavelet features**. These are reproducible choices based on the
families described in the text, not verified transcriptions of the supplement.

`normalized_geometric_length` is named for the implemented formula; it is not falsely
identified as the authors' unspecified fractal-length formula. Every feature name is exported.
At 128 channels the bundled bank has 19,456 candidates, versus the paper's 24,448.

In [ ]:
def safe_ratio(a, b):
    return a / np.maximum(np.abs(b), 1e-12)


def statistics25(x):
    """25 defined temporal descriptors, vectorized over leading dimensions."""
    x = np.asarray(x, np.float64)
    d = np.diff(x, axis=-1)
    dd = np.diff(d, axis=-1)
    mean, std = x.mean(axis=-1), x.std(axis=-1)
    absolute = np.abs(x)
    rms = np.sqrt(np.mean(x*x, axis=-1))
    variance = std**2
    mobility = np.sqrt(safe_ratio(np.var(d, axis=-1), variance))
    complexity = safe_ratio(np.sqrt(safe_ratio(np.var(dd, axis=-1), np.var(d, axis=-1))), mobility)
    # Central moments are explicitly zero for constant signals.
    standardized = (x - mean[..., None]) / np.maximum(std[..., None], 1e-12)
    values = [mean, std, variance, np.min(x, axis=-1), np.max(x, axis=-1),
              np.ptp(x, axis=-1), np.median(x, axis=-1),
              np.percentile(x, 75, axis=-1)-np.percentile(x, 25, axis=-1),
              absolute.mean(axis=-1), rms, np.sum(x*x, axis=-1),
              np.mean(standardized**3, axis=-1),
              np.where(std > 1e-12, np.mean(standardized**4, axis=-1)-3, 0),
              np.mean(np.abs(x-mean[..., None]), axis=-1),
              np.median(np.abs(x-np.median(x, axis=-1)[..., None]), axis=-1),
              np.mean(np.diff(np.signbit(x), axis=-1), axis=-1),
              np.sum(np.abs(d), axis=-1), np.mean(np.abs(d), axis=-1),
              np.mean(d*d, axis=-1), mobility, complexity,
              safe_ratio(np.max(absolute, axis=-1), rms),
              safe_ratio(rms, absolute.mean(axis=-1)),
              safe_ratio(np.max(absolute, axis=-1), absolute.mean(axis=-1)),
              safe_ratio(np.max(absolute, axis=-1), np.mean(np.sqrt(absolute), axis=-1)**2)]
    names = ["mean", "std", "variance", "min", "max", "range", "median", "iqr",
             "mean_absolute", "rms", "energy", "skewness", "excess_kurtosis",
             "mean_absolute_deviation", "median_absolute_deviation", "zero_crossing_rate",
             "waveform_length", "mean_abs_difference", "mean_squared_difference",
             "hjorth_mobility", "hjorth_complexity", "crest_factor", "shape_factor",
             "impulse_factor", "clearance_factor"]
    return np.stack(values, axis=-1), names


def frequency24(x, sfreq, high):
    freq, psd = signal.welch(x, fs=sfreq, nperseg=min(256, x.shape[-1]), axis=-1)
    keep = (freq >= 0.5) & (freq <= high)
    f, p = freq[keep], psd[..., keep]
    if len(f) < 3:
        raise ValueError("Too few frequency bins")
    prob = safe_ratio(p, p.sum(axis=-1)[..., None])
    center = (prob*f).sum(axis=-1)
    width = np.sqrt((prob*(f-center[..., None])**2).sum(axis=-1))
    cum = np.cumsum(prob, axis=-1)
    roll = lambda q: f[np.argmax(cum >= q, axis=-1)]
    entropy = -np.sum(prob*np.log(np.maximum(prob, 1e-300)), axis=-1)/np.log(len(f))
    flatness = safe_ratio(np.exp(np.log(np.maximum(p, 1e-300)).mean(axis=-1)), p.mean(axis=-1))
    peak = p.max(axis=-1)
    weighted_std = np.maximum(width, 1e-12)[..., None]
    centered_power = p-p.mean(axis=-1)[..., None]
    sd_power = np.maximum(p.std(axis=-1), 1e-12)[..., None]
    power_z = centered_power/sd_power
    values = [trapezoid(p, f, axis=-1), f[p.argmax(axis=-1)], center, width,
              roll(0.5), roll(0.85), roll(0.95), entropy, flatness,
              safe_ratio(peak, p.mean(axis=-1)),
              (prob*((f-center[..., None])/weighted_std)**3).sum(axis=-1),
              (prob*((f-center[..., None])/weighted_std)**4).sum(axis=-1),
              np.mean(power_z**3, axis=-1), np.mean(power_z**4, axis=-1)-3]
    names = ["total_power", "peak_frequency", "centroid", "bandwidth", "median_frequency",
             "rolloff85", "rolloff95", "entropy", "flatness", "crest",
             "frequency_skewness", "frequency_kurtosis", "psd_skewness", "psd_excess_kurtosis"]
    total = trapezoid(p, f, axis=-1)
    for name, lo, hi in [("delta", .5, 4), ("theta", 4, 8), ("alpha", 8, 13),
                         ("beta", 13, 30), ("gamma", 30, high)]:
        mask = (freq >= lo) & (freq < hi if hi < high else freq <= hi)
        power = trapezoid(psd[..., mask], freq[mask], axis=-1)
        values.extend([power, safe_ratio(power, total)])
        names.extend([f"{name}_power", f"{name}_relative_power"])
    return np.stack(values, axis=-1), names


def wavelet_features(x, wavelet, level):
    if pywt.dwt_max_level(x.shape[-1], pywt.Wavelet(wavelet).dec_len) < level:
        raise ValueError("Window is too short for the requested DWT level")
    coefficients = pywt.wavedec(x, wavelet, level=level, axis=-1, mode="symmetric")
    labels = [f"A{level}"] + [f"D{i}" for i in range(level, 0, -1)]
    values, names = [], []
    for band, c in zip(labels, coefficients):
        d = np.diff(c, axis=-1)
        std = c.std(axis=-1)
        z = (c-c.mean(axis=-1)[..., None])/np.maximum(std[..., None], 1e-12)
        probability = safe_ratio(c*c, np.sum(c*c, axis=-1)[..., None])
        # Geometric length of amplitude-normalized coefficients; definition explicit
        # because the paper's "fractal length" formula is unavailable.
        norm = safe_ratio(c-c.min(axis=-1)[..., None], np.ptp(c, axis=-1)[..., None])
        geometric_length = np.sum(np.sqrt(np.diff(norm, axis=-1)**2 + (1/max(c.shape[-1]-1, 1))**2), axis=-1)
        band_values = [c.mean(axis=-1), np.mean(np.abs(c), axis=-1), std, std**2,
                       np.sqrt(np.mean(c*c, axis=-1)), np.mean(c*c, axis=-1),
                       np.sum(c*c, axis=-1), np.sum(np.abs(d), axis=-1),
                       np.ptp(c, axis=-1), np.mean(z**3, axis=-1),
                       np.where(std > 1e-12, np.mean(z**4, axis=-1)-3, 0),
                       -np.sum(probability*np.log(np.maximum(probability, 1e-300)), axis=-1),
                       geometric_length]
        band_names = ["mean", "mean_absolute", "std", "variance", "rms", "mean_power",
                      "energy", "waveform_length", "range", "skewness", "excess_kurtosis",
                      "energy_entropy", "normalized_geometric_length"]
        values.extend(band_values)
        names.extend([f"{band}_{n}" for n in band_names])
    return np.stack(values, axis=-1), names


def documented_features(trial, cfg):
    """50 TD + 24 FD + 78 TFD = 152/channel; deliberately not padded to 191."""
    td, td_names = statistics25(trial)
    env, env_names = statistics25(np.abs(signal.hilbert(trial, axis=-1)))
    fd, fd_names = frequency24(trial, cfg.expected_sfreq, cfg.spectral_high)
    tf, tf_names = wavelet_features(trial, cfg.wavelet, cfg.wavelet_level)
    matrix = np.concatenate([td, env, fd, tf], axis=-1)
    names = ([f"TD_signal_{n}" for n in td_names] + [f"TD_envelope_{n}" for n in env_names]
             + [f"FD_{n}" for n in fd_names] + [f"TFD_{n}" for n in tf_names])
    if matrix.shape[-1] != 152 or len(set(names)) != 152:
        raise AssertionError("Feature schema changed")
    if not np.isfinite(matrix).all():
        raise ValueError("Non-finite features; do not silently replace numerical failures")
    return matrix, names


def prepare_features(clean, meta, directory, cfg):
    if cfg.feature_backend == "author_191_npz":
        path = Path(cfg.author_feature_dir) / f"sub-{int(meta.subject.iloc[0]):02d}_features.npz"
        with np.load(path, allow_pickle=False) as z:
            required = {"features", "feature_names", "channel_names", "trial_ids", "labels", "provenance_json"}
            if not required.issubset(z.files):
                raise ValueError(f"Author archive must contain {required}")
            feature_names = z["feature_names"].astype(str).tolist()
            author_channels = z["channel_names"].astype(str).tolist()
            author_ids = z["trial_ids"].astype(str).tolist()
            if len(set(author_ids)) != len(author_ids) or set(author_ids) != set(meta.trial_id):
                raise ValueError("Author feature trial IDs do not match loaded EEG trials")
            import mne
            canonical = mne.channels.make_standard_montage("biosemi128").ch_names
            if len(set(author_channels)) != 128 or set(author_channels) != set(canonical):
                raise ValueError("Author feature channels must identify all 128 BioSemi electrodes")
            tensor = np.asarray(z["features"], np.float64)
            if tensor.shape != (len(meta), 128, 191) or len(feature_names) != 191 or len(set(feature_names)) != 191:
                raise ValueError("Author features must be [n_trials,128,191] with 191 unique definitions")
            order = [author_ids.index(t) for t in meta.trial_id]
            if not np.array_equal(z["labels"][order], meta.y_true.to_numpy()):
                raise ValueError("Author feature labels do not match EEG labels")
            tensor = tensor[order][:, [author_channels.index(c) for c in canonical]]
            source_details = json.loads(str(z["provenance_json"].item()))
        if not np.isfinite(tensor).all():
            raise ValueError("Author feature archive contains non-finite values")
        digest = file_hash(path)
        atomic_json({"source": str(path), "sha256": digest, "details": source_details}, directory / "author_features.json")
        return tensor, feature_names, digest
    identity = {"backend": cfg.feature_backend, "wavelet": cfg.wavelet, "level": cfg.wavelet_level,
                "sfreq": cfg.expected_sfreq, "high": cfg.spectral_high, "source_code": implementation_hash()}
    digest = signature(identity)
    path = directory / f"features_{digest}.npz"
    if path.exists() and cfg.resume:
        with np.load(path, allow_pickle=False) as z:
            return z["features"], z["names"].astype(str).tolist(), digest
    tensors, names = [], None
    for trial in tqdm(clean, desc="Documented feature reconstruction (152/channel)"):
        features, names = documented_features(trial, cfg)
        tensors.append(features)
    tensor = np.stack(tensors)
    atomic_npz(path, features=tensor, names=np.asarray(names))
    atomic_csv(pd.DataFrame({"feature": names, "domain": [n.split("_")[0] for n in names],
                             "status": "reconstructed_definition_not_verified_against_T1_T3"}),
               directory / "feature_manifest.csv")
    return tensor, names, digest

## MRMR quotient selection and fresh classical models

`selection_scope="fold"` fits selection and scaling exclusively on each outer training set.
`tuning="fixed"` uses the documented defaults in `model_spec`; they are implementation
assumptions, not settings recovered from the paper. `tuning="nested"` adds inner CV and
refits feature selection inside each inner training split.

An optional `diagnostic_pre_cv` mode fits MRMR to the whole subject before CV. This sees
test labels and yields an optimistic diagnostic. It is **off by default**, is clearly labeled
in every output, and must not be reported as an unbiased test score. Its existence does not
imply the authors used this procedure.

In [ ]:
class MRMRQuotient(TransformerMixin, BaseEstimator):
    """Paper Eq.1 form: relevance / mean absolute correlation to selected features.

    MI and absolute Pearson correlation are explicit implementation choices.
    All candidates are considered; no relevance prefilter and no p-by-p matrix.
    """
    def __init__(self, k=590, relevance="mi", corr_floor=1e-6, seed=42):
        self.k, self.relevance, self.corr_floor, self.seed = k, relevance, corr_floor, seed

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        if len(X) != len(y) or not np.isfinite(X).all():
            raise ValueError("Invalid MRMR training input")
        self.n_features_in_ = X.shape[1]
        sd = X.std(axis=0)
        available_columns = np.flatnonzero(sd > 1e-12)
        if not len(available_columns):
            raise ValueError("All feature columns are constant")
        Z = (X[:, available_columns] - X[:, available_columns].mean(axis=0)) / sd[available_columns]
        if self.relevance == "mi":
            rel = mutual_info_classif(Z, y, random_state=self.seed)
        elif self.relevance == "f_stat":
            rel = f_classif(Z, y)[0]
        else:
            raise ValueError("Unknown MRMR relevance estimator")
        rel = np.nan_to_num(rel, nan=0, posinf=1e12, neginf=0)
        remaining = np.ones(len(rel), dtype=bool)
        redundancy_sum = np.zeros(len(rel), dtype=np.float64)
        chosen = []
        for step in range(min(int(self.k), len(rel))):
            score = rel.copy() if step == 0 else rel / np.maximum(redundancy_sum / step, self.corr_floor)
            score[~remaining] = -np.inf
            nxt = int(np.argmax(score))
            chosen.append(nxt)
            remaining[nxt] = False
            redundancy_sum += np.abs(Z.T @ Z[:, nxt] / len(Z))
        self.selected_indices_ = available_columns[np.asarray(chosen)]
        self.relevance_values_ = rel
        return self

    def transform(self, X):
        return np.asarray(X)[:, self.selected_indices_]


class BootstrapLR(ClassifierMixin, BaseEstimator):
    """Five independently stratified bootstraps + mean predicted probabilities.

    This follows the journal Methods description of soft voting. It is not a
    learned meta-classifier despite the paper also using the word 'stacking'.
    """
    def __init__(self, n_members=5, seed=42):
        self.n_members, self.seed = n_members, seed

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]
        self.models_, self.bootstrap_indices_ = [], []
        specs = [(0.1, "l2", "lbfgs"), (1., "l2", "lbfgs"), (10., "l2", "lbfgs"),
                 (1., "l1", "liblinear"), (1., "elasticnet", "saga")]
        for i, (C, penalty, solver) in enumerate(specs[:self.n_members]):
            rng = np.random.default_rng(self.seed + i)
            indices = np.concatenate([rng.choice(np.flatnonzero(y == c), size=np.sum(y == c), replace=True)
                                      for c in self.classes_])
            rng.shuffle(indices)
            options = {"C": C, "penalty": penalty, "solver": solver,
                       "max_iter": 5000, "random_state": self.seed + i}
            if penalty == "elasticnet":
                options["l1_ratio"] = 0.5
            model = OneVsRestClassifier(LogisticRegression(**options))
            model.fit(X[indices], y[indices])
            self.models_.append(model)
            self.bootstrap_indices_.append(indices)
        return self

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models_], axis=0)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


def model_spec(name, seed, cfg):
    if name == "svm":
        return SVC(kernel="rbf", C=1., gamma="scale", cache_size=1024), [
            {"kernel": ["linear"], "C": [.1, 1., 10.]},
            {"kernel": ["rbf"], "C": [.1, 1., 10.], "gamma": ["scale", .001, .01]}]
    if name == "lda":
        return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"), [{"shrinkage": ["auto", .1, .5]}]
    if name in {"etc", "rf"}:
        cls = ExtraTreesClassifier if name == "etc" else RandomForestClassifier
        return cls(n_estimators=500, random_state=seed, n_jobs=cfg.n_jobs), [
            {"max_depth": [None, 8], "min_samples_leaf": [1, 3]}]
    if name == "lr":
        return LogisticRegression(C=1., solver="lbfgs", max_iter=5000, random_state=seed), [{"C": [.1, 1., 10.]}]
    if name == "lgbm":
        from lightgbm import LGBMClassifier
        return LGBMClassifier(n_estimators=300, num_leaves=15, learning_rate=.05, verbosity=-1,
                              random_state=seed, n_jobs=cfg.n_jobs, deterministic=True,
                              force_col_wise=True), [{"num_leaves": [7, 15, 31], "max_depth": [-1, 5]}]
    if name in {"lr3", "lr5"}:
        return BootstrapLR(n_members=int(name[-1]), seed=seed), [{}]
    if name == "mixed3":
        return VotingClassifier([
            ("svm", SVC(C=1., kernel="rbf", probability=True, random_state=seed)),
            ("etc", ExtraTreesClassifier(n_estimators=500, random_state=seed, n_jobs=cfg.n_jobs)),
            ("lr", LogisticRegression(max_iter=5000, random_state=seed))], voting="soft"), [{}]
    raise ValueError(f"Unknown classical model: {name}")


def fit_feature_transform(X, y, cfg, seed, select=True, cache=False):
    # Keep non-finite imputation and scaling local to training. The normal feature
    # backend refuses non-finite values; imputer also supports external archives.
    cache_path = None
    if cache:
        identity = {"X": hashlib.sha256(np.ascontiguousarray(X, dtype=np.float64).tobytes()).hexdigest(),
                    "y": hashlib.sha256(np.ascontiguousarray(y, dtype=np.int64).tobytes()).hexdigest(),
                    "shape": X.shape, "seed": seed, "select": select, "k": cfg.k_features,
                    "relevance": cfg.mrmr_relevance, "floor": cfg.mrmr_corr_floor,
                    "code": implementation_hash(), "versions": versions()}
        cache_path = Path(cfg.output_root)/"feature_transforms"/(signature(identity)+".pkl")
        if cfg.resume and cache_path.exists():
            return load_ml_checkpoint(cache_path)
    steps = [("imputer", SimpleImputer(strategy="median", keep_empty_features=True))]
    if select:
        steps.append(("mrmr", MRMRQuotient(cfg.k_features, cfg.mrmr_relevance, cfg.mrmr_corr_floor, seed)))
    steps.append(("scaler", StandardScaler()))
    transformer = Pipeline(steps).fit(X, y)
    if cache_path is not None:
        dump_bundle(transformer, cache_path)
    return transformer


def checked_splits(y, n_splits, seed):
    classes, counts = np.unique(y, return_counts=True)
    if not np.array_equal(classes, np.arange(4)):
        raise ValueError("Every subject must contain all four classes")
    if counts.min() < n_splits:
        raise ValueError(f"Requested {n_splits} folds but smallest class has {counts.min()} trials")
    # No silent fold-count reduction: a 2-fold smoke test cannot masquerade as 10-fold.
    return list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))


def select_ml_parameters(X, y, name, cfg, seed, select=True):
    base, grid = model_spec(name, seed, cfg)
    candidates = list(ParameterGrid(grid))
    if cfg.tuning == "fixed" or len(candidates) == 1:
        return base, {}, pd.DataFrame()
    scores = np.zeros((cfg.inner_folds, len(candidates)))
    for fold, (train, val) in enumerate(checked_splits(y, cfg.inner_folds, seed)):
        transform = fit_feature_transform(X[train], y[train], cfg, seed + fold, select=select, cache=True)
        A, B = transform.transform(X[train]), transform.transform(X[val])
        for j, parameters in enumerate(candidates):
            estimator = clone(base).set_params(**parameters).fit(A, y[train])
            scores[fold, j] = balanced_accuracy_score(y[val], estimator.predict(B))
    means = scores.mean(axis=0)
    best = candidates[int(np.argmax(means))]
    frame = pd.DataFrame({"parameters": [json.dumps(c, sort_keys=True) for c in candidates],
                          "inner_balanced_accuracy": means})
    return clone(base).set_params(**best), best, frame


def dump_bundle(bundle, path):
    # cloudpickle serializes notebook-defined classes for reuse in a fresh kernel.
    import cloudpickle
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(".tmp")
    with temp.open("wb") as f:
        cloudpickle.dump(bundle, f)
    temp.replace(path)


def fit_ml_fold(X, y, train, test, model_name, cfg, fold_seed, directory, names,
                pre_cv_selector=None):
    # The selector is only pre-fitted when the explicitly optimistic diagnostic is enabled.
    if pre_cv_selector is not None:
        X_input = pre_cv_selector.transform(X)
    else:
        X_input = X
    estimator, selected_params, cv_frame = select_ml_parameters(
        X_input[train], y[train], model_name, cfg, fold_seed, select=pre_cv_selector is None)
    transformer = fit_feature_transform(X_input[train], y[train], cfg, fold_seed, select=pre_cv_selector is None, cache=True)
    A, B = transformer.transform(X_input[train]), transformer.transform(X_input[test])
    estimator.fit(A, y[train])  # new estimator, fitted from scratch on outer training only
    pred = estimator.predict(B)
    selected = (pre_cv_selector.selected_indices_ if pre_cv_selector is not None
                else transformer.named_steps["mrmr"].selected_indices_)
    atomic_csv(pd.DataFrame({"rank": np.arange(1, len(selected)+1), "feature": np.asarray(names)[selected],
                             "column_index": selected}), directory / "selected_features.csv")
    if len(cv_frame):
        atomic_csv(cv_frame, directory / "inner_cv.csv")
    dump_bundle({"pre_cv_selector": pre_cv_selector, "transformer": transformer, "model": estimator,
                 "feature_names": names, "config": asdict(cfg), "versions": versions(),
                 "train_indices": train, "test_indices": test}, directory / "model.pkl")
    return pred, {"best_parameters": selected_params, "n_selected": len(selected)}

## Deep model reconstructions: 1D CNN and feature-image ResNet18/VGG16

The outer training set is split internally to choose the epoch count without using the
outer test set. The network is then initialized again and fitted to all outer training
trials for that many epochs. Normalization is refitted on those training trials.
This epoch-selection procedure is an explicit addition because the paper does not give
enough training detail. Image variants are created **after** the original trial split.
ImageNet weights are disabled by default because their use is not established by the paper.
VGG16 is memory intensive: use batch size 4–8 if a T4 runs out of memory.

In [ ]:
def make_deep_model(name, n_channels, n_times, cfg):
    import torch
    from torch import nn
    if name == "cnn1d":
        # k=3, pool=4, conv dropout=.1 and MLP dropout=.5 follow the paper.
        # Widths and hidden dimension are disclosed assumptions pending Figure S2.
        a, b = cfg.cnn_widths
        return nn.Sequential(
            nn.Conv1d(n_channels, a, 3, padding=1), nn.ReLU(), nn.MaxPool1d(4), nn.Dropout(.1),
            nn.Conv1d(a, b, 3, padding=1), nn.ReLU(), nn.MaxPool1d(4), nn.Dropout(.1),
            nn.Flatten(), nn.Linear(b*(n_times//16), cfg.cnn_hidden), nn.ReLU(),
            nn.Dropout(.5), nn.Linear(cfg.cnn_hidden, 4))
    from torchvision import models
    if name == "resnet18":
        net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if cfg.image_pretrained else None)
        net.fc = nn.Linear(net.fc.in_features, 4)
    elif name == "vgg16":
        net = models.vgg16(weights=models.VGG16_Weights.DEFAULT if cfg.image_pretrained else None)
        net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, 4)
    else:
        raise ValueError(name)
    return net


def deep_arrays(base, train, cfg, image=False):
    """Statistics fitted exclusively on the specified training indices."""
    if not image:  # [trial,channel,time]
        mean = base[train].mean(axis=(0, 2), keepdims=True)
        std = np.maximum(base[train].std(axis=(0, 2), keepdims=True), 1e-8)
        data = ((base-mean)/std).astype(np.float32)
    else:  # [trial,channel,feature] -> feature-by-channel images
        import torch
        from torch.nn import functional as F
        mean = base[train].mean(axis=(0, 1), keepdims=True)
        std = np.maximum(base[train].std(axis=(0, 1), keepdims=True), 1e-8)
        grey = np.clip((base-mean)/std, -5, 5).transpose(0, 2, 1).copy()/5
        data = F.interpolate(torch.from_numpy(grey.astype(np.float32))[:, None],
                             size=(cfg.image_size, cfg.image_size), mode="bilinear",
                             align_corners=False).numpy()  # one grey channel; expand in model input
    return data, {"mean": mean, "std": std, "image": image}


def make_deep_loader(data, y, indices, cfg, training, image, seed):
    import torch
    from torch.utils.data import DataLoader, Dataset
    class Trials(Dataset):
        def __len__(self):
            return len(indices)*(8 if training and image else 1)

        def __getitem__(self, item):
            base_index = int(indices[item % len(indices)])
            x = torch.from_numpy(data[base_index].copy())
            if image:
                x = x.expand(3, -1, -1).clone()
                # Original + 3 rotations + 4 translations from the paper.
                # All variants of a trial stay in its original training partition.
                variant = item // len(indices) if training else 0
                if 1 <= variant <= 3:
                    x = torch.rot90(x, variant, dims=(-2, -1))
                elif variant >= 4:
                    from torchvision.transforms.functional import affine
                    delta = round(cfg.image_size*.1)
                    shifts = [(0, -delta), (delta, 0), (delta, -delta), (-delta, delta)]
                    x = affine(x, angle=0., translate=list(shifts[variant-4]), scale=1., shear=[0., 0.], fill=0.)
                if cfg.image_pretrained:
                    x = (x+1)/2
                    x = (x-torch.tensor([.485,.456,.406])[:,None,None])/torch.tensor([.229,.224,.225])[:,None,None]
            return x, torch.tensor(int(y[base_index]), dtype=torch.long)
    return DataLoader(Trials(), batch_size=cfg.deep_batch_size, shuffle=training, num_workers=0,
                      generator=torch.Generator().manual_seed(seed))


def train_network(model, loader, device, epochs, cfg, validation_loader=None):
    import torch
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.deep_lr)
    criterion = torch.nn.CrossEntropyLoss()
    best, best_epoch, stale, history = -np.inf, 1, 0, []
    for epoch in range(1, epochs+1):
        model.train()
        loss_sum, correct, count = 0., 0, 0
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(Xb)
            loss = criterion(logits, yb)
            if not torch.isfinite(loss):
                raise ValueError("Non-finite neural-network loss")
            loss.backward()
            optimizer.step()
            loss_sum += loss.item()*len(yb)
            correct += (logits.argmax(1)==yb).sum().item()
            count += len(yb)
        row = {"epoch": epoch, "train_loss": loss_sum/count, "train_accuracy": correct/count}
        if validation_loader is not None:
            truth, pred = predict_network(model, validation_loader, device)
            score = balanced_accuracy_score(truth, pred)
            row["validation_balanced_accuracy"] = score
            if score > best + 1e-12:
                best, best_epoch, stale = score, epoch, 0
            else:
                stale += 1
        history.append(row)
        if validation_loader is not None and stale >= cfg.deep_patience:
            break
    return best_epoch if validation_loader is not None else epochs, pd.DataFrame(history)


def predict_network(model, loader, device):
    import torch
    model.eval()
    truth, predictions = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            predictions.extend(model(Xb.to(device)).argmax(1).cpu().numpy().tolist())
            truth.extend(yb.numpy().tolist())
    return np.asarray(truth), np.asarray(predictions)


def fit_deep_fold(base, y, train, test, model_name, cfg, fold_seed, directory):
    import torch
    image = model_name != "cnn1d"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(fold_seed)
    val_size = max(4, int(np.ceil(len(train)*cfg.deep_validation_fraction)))
    inner_train, inner_val = next(StratifiedShuffleSplit(n_splits=1, test_size=val_size,
                                                        random_state=fold_seed).split(base[train], y[train]))
    fitting, validation = train[inner_train], train[inner_val]
    data, _ = deep_arrays(base, fitting, cfg, image)
    net = make_deep_model(model_name, base.shape[1], base.shape[-1], cfg).to(device)
    train_loader = make_deep_loader(data, y, fitting, cfg, True, image, fold_seed)
    val_loader = make_deep_loader(data, y, validation, cfg, False, image, fold_seed)
    chosen_epochs, history = train_network(net, train_loader, device, cfg.deep_epochs, cfg, val_loader)
    atomic_csv(history, directory / "epoch_selection_history.csv")
    del net, data, train_loader, val_loader
    if device.type == "cuda":
        torch.cuda.empty_cache()
    # Reinitialize and train on all nine outer-training folds for selected epochs.
    # The outer test fold has never affected epoch selection or normalization.
    set_seed(fold_seed)
    data, preprocessing = deep_arrays(base, train, cfg, image)
    net = make_deep_model(model_name, base.shape[1], base.shape[-1], cfg).to(device)
    loader = make_deep_loader(data, y, train, cfg, True, image, fold_seed)
    _, refit_history = train_network(net, loader, device, chosen_epochs, cfg)
    truth, predictions = predict_network(net, make_deep_loader(data, y, test, cfg, False, image, fold_seed), device)
    if not np.array_equal(truth, y[test]):
        raise AssertionError("Deep test predictions lost trial ordering")
    atomic_csv(refit_history, directory / "refit_history.csv")
    atomic_npz(directory / "normalization.npz", mean=preprocessing["mean"], std=preprocessing["std"])
    torch.save({"state_dict": {k: v.detach().cpu() for k,v in net.state_dict().items()},
                "config": asdict(cfg), "model_name": model_name, "n_channels": base.shape[1],
                "n_times": base.shape[-1], "chosen_epochs": chosen_epochs,
                "train_indices": train.tolist(), "test_indices": test.tolist(),
                "normalization": "normalization.npz"}, directory / "model.pt")
    del net, data, loader
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return predictions, {"chosen_epochs": chosen_epochs, "device": str(device),
                          "n_inner_train": len(fitting), "n_inner_validation": len(validation)}

## Evaluation, fold resume and exact subject-level out-of-fold metrics

In [ ]:
def completed_fold(directory, expected_ids, expected_labels, is_deep, resume):
    files = [directory / "complete.json", directory / "predictions.csv",
             directory / ("model.pt" if is_deep else "model.pkl")]
    if not resume or not all(p.exists() for p in files):
        return None
    if json.loads((directory / "complete.json").read_text()).get("status") != "complete":
        return None
    frame = pd.read_csv(directory / "predictions.csv")
    if (frame.trial_id.tolist() != list(expected_ids)
            or not np.array_equal(frame.y_true.to_numpy(), expected_labels)):
        raise ValueError(f"Stale checkpoint predictions: {directory}")
    return frame


def aggregate_oof(oof):
    group = ["protocol", "feature_backend", "model", "n_channels", "subject", "seed"]
    rows = []
    for key, frame in oof.groupby(group, dropna=False):
        if frame.trial_id.duplicated().any():
            raise AssertionError("Duplicate OOF prediction within a subject/model/seed")
        row = dict(zip(group, key))
        row.update(n_trials=len(frame), **metrics(frame.y_true, frame.y_pred))
        rows.append(row)
    subjects = pd.DataFrame(rows)
    # First average repeated seeds within each subject; SD measures between-subject
    # variation rather than treating seeds as independent participants.
    subject_keys = group[:-1]
    metric_names = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
    seed_averaged = subjects.groupby(subject_keys, as_index=False)[metric_names].mean()
    overall_keys = group[:-2]
    summary = seed_averaged.groupby(overall_keys)[metric_names].agg(["mean", "std"])
    summary.columns = [f"{a}_{b}" for a,b in summary.columns]
    summary = summary.reset_index()
    counts = seed_averaged.groupby(overall_keys).size().rename("n_subjects").reset_index()
    summary = summary.merge(counts, on=overall_keys)
    return subjects, seed_averaged, summary


def evaluate_subject(clean, tensor, feature_names, names, meta, cfg, run_dir, data_identity):
    cfg.validate()
    sid = int(meta.subject.iloc[0])
    y = meta.y_true.to_numpy(dtype=int)
    subsets = channel_subsets(names)
    frames = []
    for nch in cfg.channel_counts:
        indices = subsets[nch]
        features = tensor[:, indices].reshape(len(tensor), -1)
        flat_names = [f"{names[ch]}::{f}" for ch in indices for f in feature_names]
        for seed in cfg.seeds:
            pre_cv = None
            if cfg.selection_scope == "diagnostic_pre_cv":
                warnings.warn("OPTIMISTIC DIAGNOSTIC: feature selection sees outer-test labels. Do not report as unbiased accuracy.")
                pre_cv = MRMRQuotient(cfg.k_features, cfg.mrmr_relevance, cfg.mrmr_corr_floor, seed).fit(features, y)
            for model_name in cfg.ml_models + cfg.deep_models:
                is_deep = model_name in cfg.deep_models
                image = model_name in {"resnet18", "vgg16"}
                nfolds = cfg.image_outer_folds if image else cfg.outer_folds
                outer = checked_splits(y, nfolds, seed)
                model_frames = []
                for fold, (train, test) in enumerate(outer, start=1):
                    fold_seed = int(seed*100000+sid*100+fold)
                    protocol = ("subject_specific" if cfg.selection_scope == "fold" else "OPTIMISTIC_pre_cv_diagnostic")
                    training_config = asdict(cfg)
                    for operational in ("data_root", "output_root", "subject_ids", "channel_counts", "seeds",
                                        "ml_models", "deep_models", "resume", "n_jobs"):
                        training_config.pop(operational)
                    identity = {"code": implementation_hash(), "config": training_config, "data": data_identity,
                                "model": model_name, "subject": sid, "seed": seed, "n_channels": nch,
                                "channel_names": [names[i] for i in indices], "fold": fold,
                                "train_ids": meta.trial_id.iloc[train].tolist(),
                                "test_ids": meta.trial_id.iloc[test].tolist(), "versions": versions()}
                    digest = signature(identity)
                    directory = Path(cfg.output_root) / "checkpoints" / f"{model_name}_{nch}ch_sub-{sid:02d}_seed-{seed}" / f"fold-{fold:02d}_{digest}"
                    directory.mkdir(parents=True, exist_ok=True)
                    frame = completed_fold(directory, meta.trial_id.iloc[test], y[test], is_deep, cfg.resume)
                    if frame is None:
                        atomic_json(identity, directory / "config.json")
                        start = time.monotonic()
                        if is_deep:
                            base = tensor[:, indices] if image else clean[:, indices]
                            pred, details = fit_deep_fold(base, y, train, test, model_name, cfg, fold_seed, directory)
                        else:
                            pred, details = fit_ml_fold(features, y, train, test, model_name, cfg, fold_seed,
                                                        directory, flat_names, pre_cv)
                        frame = meta.iloc[test].copy().reset_index(drop=True)
                        frame["y_pred"] = pred
                        frame = frame.assign(model=model_name, n_channels=nch, seed=seed, outer_fold=fold,
                                             outer_folds=nfolds, protocol=protocol, feature_backend=cfg.feature_backend)
                        atomic_csv(frame, directory / "predictions.csv")
                        atomic_json({"status": "complete", "signature": digest,
                                     "elapsed_seconds": time.monotonic()-start,
                                     "metrics": metrics(y[test], pred), "details": details}, directory / "complete.json")
                    print(f"{model_name:9s} | subject {sid:02d} | {nch:3d} ch | fold {fold:02d}/{nfolds} | accuracy={accuracy_score(frame.y_true, frame.y_pred):.3f}")
                    model_frames.append(frame)
                combined = pd.concat(model_frames, ignore_index=True)
                if set(combined.trial_id) != set(meta.trial_id) or combined.trial_id.duplicated().any():
                    raise AssertionError("OOF coverage failed: every trial must appear exactly once")
                frames.append(combined)
                # Persist as each subject/model completes; final scoring is OOF, not fold mean.
                atomic_csv(combined, Path(run_dir)/"subject_oof"/f"{model_name}_{nch}ch_sub-{sid:02d}_seed-{seed}.csv")
                cm = confusion_matrix(combined.y_true, combined.y_pred, labels=range(4))
                atomic_csv(pd.DataFrame(cm, columns=CLASS_NAMES).assign(true_class=CLASS_NAMES),
                           Path(run_dir)/"confusion_matrices"/f"{model_name}_{nch}ch_sub-{sid:02d}_seed-{seed}.csv")
    if not frames:
        raise ValueError("Select at least one ML or deep model")
    return pd.concat(frames, ignore_index=True)


def run_paper_methods(cfg):
    cfg.validate()
    records = discover_recordings(cfg.data_root)
    available = sorted({r[0] for r in records})
    subjects = available if cfg.subject_ids is None else list(cfg.subject_ids)
    if not set(subjects).issubset(available):
        raise ValueError(f"Requested subjects not in dataset: {set(subjects)-set(available)}")
    run_identity = {"method": METHOD_VERSION, "code": implementation_hash(), "config": asdict(cfg),
                    "versions": versions(), "subjects": subjects}
    # Actual source content digests enter fold identities after loading each subject.
    directory = Path(cfg.output_root)/"runs"/signature(run_identity)
    directory.mkdir(parents=True, exist_ok=True)
    atomic_json(run_identity, directory/"run_config.json")
    atomic_json({"exact_reproduction": False, "unresolved": MISSING_AUTHOR_DETAILS,
                 "feature_backend": cfg.feature_backend,
                 "note": "Author feature archives do not by themselves resolve training/preprocessing ambiguity."},
                directory/"reproduction_status.json")
    print("Results:", directory)
    print("Reconstruction with explicit assumptions; exact reproduction remains unverified.")
    all_frames = []
    for subject in subjects:
        raw, times, names, meta, provenance = load_subject(cfg, subject)
        clean, cache_dir, audit = prepare_clean_subject(raw, times, names, meta, provenance, cfg)
        del raw
        tensor, feature_names, feature_identity = prepare_features(clean, meta, cache_dir, cfg)
        mapping = {str(n): [names[i] for i in ix] for n, ix in channel_subsets(names).items()}
        atomic_json(mapping, directory/"channel_subsets.json")
        atomic_csv(meta, directory/"data_audits"/f"sub-{subject:02d}_trials.csv")
        atomic_json(provenance, directory/"data_audits"/f"sub-{subject:02d}_provenance.json")
        frame = evaluate_subject(clean, tensor, feature_names, names, meta, cfg, directory,
                                 {"signal_cache": cache_dir.name, "features": feature_identity})
        all_frames.append(frame)
        oof = pd.concat(all_frames, ignore_index=True)
        subject_metrics, seed_averaged, summary = aggregate_oof(oof)
        atomic_csv(oof, directory/"oof_predictions.csv")
        atomic_csv(subject_metrics, directory/"subject_seed_metrics.csv")
        atomic_csv(seed_averaged, directory/"subject_metrics.csv")
        atomic_csv(summary, directory/"summary.csv")
        atomic_json({"complete": len(all_frames)==len(subjects), "completed_subjects": subjects[:len(all_frames)],
                     "requested_subjects": subjects}, directory/"progress.json")
        del clean, tensor
    print(summary.to_string(index=False))
    return {"directory": directory, "oof": oof, "subjects": subject_metrics, "summary": summary}


def load_ml_checkpoint(path):
    """Load ONLY checkpoints you trust; pickle can execute code."""
    import cloudpickle
    with Path(path).open("rb") as f:
        return cloudpickle.load(f)


def predict_ml_checkpoint(bundle, features):
    """features must have the saved feature_names order and preprocessing."""
    if np.asarray(features).shape[1] != len(bundle["feature_names"]):
        raise ValueError("Input features do not match checkpoint schema")
    X = features
    if bundle["pre_cv_selector"] is not None:
        X = bundle["pre_cv_selector"].transform(X)
    return bundle["model"].predict(bundle["transformer"].transform(X))

In [ ]:
NOTEBOOK_SOURCE_SHA256 = "db0fffbc32f637c9e5fb160607c51736c2cbc6fd15d56e8ae27653f877908f9b"
print("Implementation:", METHOD_VERSION)
print("Packages:", versions())

## 2. Configure one run here

Start with `PROFILE="smoke"`: one subject, 128 channels, SVM, two outer folds. It tests
execution and is not a 10-fold result. Then use `"svm_subject1"` for a full 10-fold subject
and `"all_ml"` for all subjects and classical models. All profiles retain 128 channels.
Set `(128,64,32,16)` explicitly only after inspecting the full-channel benchmark.

The CPU work in full-pool MRMR can be substantial. `nested` multiplies that work by the
number of inner folds. A GPU will not speed up that CPU feature-selection stage. Identical
training-set feature transforms are cached and reused across the classical model families.

In [ ]:
PROFILE = "smoke"  # smoke, svm_subject1, all_ml, cnn1d, image_cnns, custom
RUN_EXPERIMENTS = False  # set True after checking DATA_ROOT
DATA_ROOT = "/kaggle/input/datasets/truthisneverlinear/inner-speech-recognition/inner-speech-recognition/derivatives"
OUTPUT_ROOT = "/kaggle/working/paper_methods"

cfg = PaperConfig(data_root=DATA_ROOT, output_root=OUTPUT_ROOT)
if PROFILE == "smoke":
    cfg.subject_ids = (1,)
    cfg.ml_models = ("svm",)
    cfg.outer_folds = 2
elif PROFILE == "svm_subject1":
    cfg.subject_ids = (1,)
    cfg.ml_models = ("svm",)
elif PROFILE == "all_ml":
    cfg.subject_ids = None
    cfg.ml_models = ("svm", "lda", "etc", "rf", "lr", "lgbm", "mixed3", "lr3", "lr5")
elif PROFILE == "cnn1d":
    cfg.subject_ids = None
    cfg.ml_models = ()
    cfg.deep_models = ("cnn1d",)
elif PROFILE == "image_cnns":
    cfg.subject_ids = None
    cfg.ml_models = ()
    cfg.deep_models = ("resnet18", "vgg16")
    cfg.deep_batch_size = 4
elif PROFILE != "custom":
    raise ValueError(PROFILE)

# Optional changes; uncomment only the settings you want.
# cfg.channel_counts = (128, 64, 32, 16)
# cfg.subject_ids = (1, 2)
# cfg.seeds = (42, 13, 101)
# cfg.tuning = "nested"
# cfg.artifact_policy = "manual"
# cfg.manual_artifact_channels = {1: ["A1", "B12"]}  # EXAMPLE ONLY, not the paper's bad channels
# cfg.feature_backend = "author_191_npz"
# cfg.author_feature_dir = "/kaggle/input/author-features"

cfg.validate()
print(json.dumps(asdict(cfg), indent=2))
print("Exact paper reproduction: UNVERIFIED. See limitations above.")

## 3. Optional inspection before training

Set `INSPECT_DATA=True` to see the source manifest, class counts, a trial waveform and
per-channel mean/std. This does not train models. The trial trace shows the configured
action interval against the actual FIF time axis. The paper uses derivatives that were
already processed by the dataset provider; do not substitute raw BDF data without the
provider's preprocessing.

In [ ]:
INSPECT_DATA = False
if INSPECT_DATA:
    import matplotlib.pyplot as plt
    recordings = discover_recordings(cfg.data_root)
    sid = cfg.subject_ids[0] if cfg.subject_ids else recordings[0][0]
    raw, times, channels, trial_metadata, provenance = load_subject(cfg, sid)
    display(pd.DataFrame(provenance))
    display(pd.crosstab(trial_metadata.session, trial_metadata.y_true))
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    axes[0].plot(times, raw[0, 0])
    axes[0].axvspan(cfg.action_start, cfg.action_stop, color="orange", alpha=.2)
    axes[0].set(xlabel="FIF time (s)", ylabel="microvolts", title=f"Trial 0, channel {channels[0]}")
    axes[1].plot(raw.mean(axis=(0,2)))
    axes[1].set(ylabel="Mean (microvolts)", xlabel="BioSemi channel index")
    axes[2].plot(raw.std(axis=(0,2)))
    axes[2].set(ylabel="Std (microvolts)", xlabel="BioSemi channel index")
    plt.tight_layout()
    plt.show()
    del raw

## 4. Easy runner

Each model/fold is independent. A completed fold is reused only when its source data,
preprocessing, feature backend, code, parameters, channel list, split and dependency
versions match. Changing the requested list of subjects/models/channel counts does not
invalidate matching existing fold checkpoints. An interrupted fold restarts from its
beginning; neural optimizer state is not resumed mid-epoch. VMD also resumes per trial.

In [ ]:
paper_results = None
if RUN_EXPERIMENTS:
    paper_results = run_paper_methods(cfg)
    display(paper_results["summary"])
else:
    print("Ready. Set DATA_ROOT, choose PROFILE, set RUN_EXPERIMENTS=True, then run this cell.")

## 5. Reading, saving and resuming results

Download the complete `paper_methods` output directory from Kaggle. CSV summaries alone
cannot restore trained models. For another session, copy this directory into the new
writable output location and set `OUTPUT_ROOT` to it. Input file content hashes are recorded;
cache paths also include provenance paths, so moving/renaming the input dataset can cause
recomputation. Kaggle input mounts are read-only; use a writable output directory.

| File | Meaning |
|---|---|
| `runs/<id>/oof_predictions.csv` | Each original trial exactly once per subject/model/channel/seed |
| `subject_seed_metrics.csv` | Metrics calculated on concatenated OOF predictions |
| `subject_metrics.csv` | Seed-averaged metrics for each participant |
| `summary.csv` | Mean and SD across participants; one subject gives undefined SD |
| `progress.json` | Shows whether all requested subjects have completed |
| `confusion_matrices/` | Four-class counts per subject and model |
| `checkpoints/.../model.pkl` | Classical model + scaler + MRMR + feature order |
| `checkpoints/.../model.pt` | Neural model weights + architecture/configuration |
| `normalization.npz` | Neural training-set normalization required for future inference |
| `selected_features.csv` | Actual MRMR-selected columns for that fold |
| `cache/.../artifact_audit.csv` | Channel flags, VMD frequencies, before/after variance |
| `feature_transforms/` | Training-data-hashed MRMR/scaler caches shared across classifiers |
| `feature_manifest.csv` | Complete list of bundled feature names and their status |
| `reproduction_status.json` | Machine-readable unresolved reproduction details |

The main `summary.csv` uses flat column names and retains all identifying columns.
Accuracy is computed from OOF trial predictions, **not an unweighted mean of fold accuracies**.
The same applies to macro/weighted F1, which cannot in general be reconstructed from fold F1.

In [ ]:
# To inspect a previously completed run without retraining:
# result_dir = Path(OUTPUT_ROOT) / "runs" / "<run-id>"
# display(pd.read_csv(result_dir / "summary.csv"))
# oof = pd.read_csv(result_dir / "oof_predictions.csv")
# subject_seed, subject_average, summary = aggregate_oof(oof)

# To reuse a trusted SVM checkpoint on features in exactly its saved column order:
# bundle = load_ml_checkpoint("/kaggle/working/paper_methods/checkpoints/.../model.pkl")
# predictions = predict_ml_checkpoint(bundle, features_for_new_trials)

## 6. Connecting the exact author feature bank

To replace the reconstruction, obtain Supplementary Tables T1–T3 and the relevant extraction
code, or the authors' extracted features and preprocessing provenance. For each subject,
save a trusted `sub-01_features.npz` (and so on) containing:

```text
features         float array [n_inner_trials,128,191]
feature_names    191 unique strings documenting definitions/order
channel_names    128 BioSemi names (validated and reordered)
trial_ids        IDs from the exported trials.csv
labels           class labels corresponding to those IDs
provenance_json  JSON string describing extraction code, units, VMD and feature definitions
```

Choose `cfg.feature_backend="author_191_npz"` and set `cfg.author_feature_dir`. The loader
checks dimensions, IDs, labels, channel identities, uniqueness and finiteness. It never
matches rows using their position alone. Author-provided arrays resolve the feature input,
but do not automatically resolve unspecified model settings or selection scope.

The missing information should not be filled in by tuning until a published score is reached.
Request the supplementary definitions, bad-channel decisions, VMD settings, MRMR package,
fold indices, classifier settings and exact preprocessing/selection order from the authors.

## Sources and verification scope

- Journal PDF supplied with this project: DOI 10.1109/ACCESS.2025.3644494, Sections II.B–H.
- [Author preprint](https://arxiv.org/abs/2412.17824).
- [Dataset authors' repository](https://github.com/N-Nieto/Inner_Speech_Dataset).
- [Dataset publication and trial timing](https://www.nature.com/articles/s41597-022-01147-2).
- [VMD Python implementation](https://github.com/vrcarva/vmdpy).
- [scikit-learn cross-validation guidance](https://scikit-learn.org/stable/modules/cross_validation.html).

The implementation is tested on synthetic data for feature counts/numerics, VMD, event
alignment, fresh folds, exact OOF aggregation, checkpoint reuse, all classical model paths,
the CNN fitting path and image-model forward passes. This is software validation, not a
real-EEG experiment. Full EEG training and T4 performance were not measured locally.